# Model Intercomparison

This notebook allows for the comparison of multiple model runs. It scans the output directories of different experiments and generates comparative plots.

## Features

*   **Metric Comparison**: Side-by-side comparison of scalar metrics (MAE, RMSE, CRPS, etc.).
*   **Vertical Profiles**: Comparison of error profiles across pressure levels.
*   **Energy Spectra**: Comparison of kinetic energy spectra.
*   **Maps**: Visual comparison of forecast fields and error maps.

## Relevant Code

The core logic resides in `src/swissclim_evaluations/intercompare.py`.

Key functions:
*   [`intercompare_vertical_profiles`](../src/swissclim_evaluations/intercompare.py#L196): Overlays vertical profile metrics from multiple models.
*   [`_scan_model_sets`](../src/swissclim_evaluations/intercompare.py#L47): Identifies common files across model output directories to ensure fair comparison.

# Model Intercomparison Demo
This notebook shows how to combine saved artifacts produced by the CLI for multiple models.

## 1. Configuration

Edit the lists below to point to the per-model evaluation output folders you want to intercompare. Optionally provide human-readable labels (must match length of `model_dirs`). If `labels` is left empty or length mismatch occurs, folder basenames are used.

In [ ]:
import yaml
from pathlib import Path
import pandas as pd
import sys
import importlib
from swissclim_evaluations.intercompare import run_from_config
from swissclim_evaluations.helpers import display_outputs

# --- 1. Configuration (via YAML) ------------------------------------------
# Resolve repo root early and ensure local package is importable
def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "pyproject.toml").exists() or (cur / ".git").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

project_root = _find_repo_root(Path.cwd())
root_dir = project_root  # expose as 'root_dir' for clarity in the notebook
sys.path.insert(0, str(project_root / "src"))  # prefer local package in src/

# YAML config path (relative to repo)
cfg_path = (project_root / "config/intercomparison.yaml").resolve()
print("Using intercomparison config:", cfg_path)

# Load YAML and compute derived paths (without running yet)
with open(cfg_path) as f:
    cfg = yaml.safe_load(f) or {}

# OVERRIDE for manual run
cfg["models"] = ["output/verification_esfm", "output/verification_esfm_perturbed"]
cfg["labels"] = ["Model A (Original)", "Model B (Perturbed)"]

# Resolve relative paths against project root for display
model_dirs = [
    (project_root / p).resolve() if not str(p).startswith("/") else Path(p)
    for p in cfg.get("models", [])
 ]
labels = cfg.get("labels") or []
if not labels or len(labels) != len(model_dirs):
    labels = [p.name for p in model_dirs]
out_root = (
    (project_root / cfg.get("output_root", "output/intercomparison")).resolve()
    if not str(cfg.get("output_root", "output/intercomparison")).startswith("/")
    else Path(cfg.get("output_root")).resolve()
)
modules = [str(m).lower() for m in (cfg.get("modules") or ["spectra","hist","kde","maps","metrics","prob","vprof"]) ]
max_map_panels = int(cfg.get("max_map_panels", 4))
max_crps_map_panels = int(cfg.get("max_crps_map_panels", 4))

# Build a normalized config anchored to the repo root so all paths are absolute
run_cfg = {
    "models": [str(p) for p in model_dirs],
    "labels": labels,
    "output_root": str(out_root),
    "modules": modules,
    "max_map_panels": max_map_panels,
    "max_crps_map_panels": max_crps_map_panels,
}

print("Models:")
for p in model_dirs:
    print(" -", p.relative_to(project_root) if str(p).startswith(str(project_root)) else p)
print("Labels:", labels)
print("Output root:", out_root.relative_to(project_root) if str(out_root).startswith(str(project_root)) else out_root)
print("Modules:", modules)

out_root.mkdir(parents=True, exist_ok=True)
model_dirs

## 2. Run Intercomparison
Runs the selected intercomparison modules based on the YAML. After the run, we render per-module results in dedicated sections: figures first, then tables.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    run_from_config(run_cfg)

## 3. Energy Spectra

Compares the kinetic energy spectra of the models.
This analysis reveals how well the models capture atmospheric variability at different spatial scales.

**Relevant Code:**
*   [`intercompare_energy_spectra`](../src/swissclim_evaluations/intercompare.py#L347): Overlays spectra from multiple models.
*   [`_scan_model_sets`](../src/swissclim_evaluations/intercompare.py#L47): Ensures only common files are compared.

In [ ]:
display_outputs(out_root / "energy_spectra", pattern_csv="")

In [ ]:
# Energy Spectra — Tables (LSD combined)
display_outputs(out_root / "energy_spectra", pattern_img="", pattern_csv="*.csv")

## 4. Histograms

Compares the distribution of values for each variable across models.
Useful for checking if models capture the correct range and frequency of values (e.g., extreme events).

**Relevant Code:**
*   [`intercompare_histograms`](../src/swissclim_evaluations/intercompare.py#L729): Generates side-by-side histogram plots.

In [ ]:
# Histograms — Figures & Tables
display_outputs(out_root / "histograms")

## 5. Wasserstein Distance & KDE

Compares the Wasserstein distance and Kernel Density Estimates (KDE) of the distributions.
This provides a quantitative and visual comparison of how close the model distributions are to the target.

**Relevant Code:**
*   [`intercompare_wd_kde`](../src/swissclim_evaluations/intercompare.py#L904): Aggregates Wasserstein distances and plots KDEs.

In [ ]:
# KDE — Figures & Tables
display_outputs(out_root / "wd_kde")

## 6. Maps

Visual comparison of forecast fields and error maps.
Allows for a qualitative assessment of spatial patterns and biases.

**Relevant Code:**
*   [`intercompare_maps`](../src/swissclim_evaluations/intercompare.py#L1052): Creates multi-panel figures showing fields from different models side-by-side.

In [ ]:
# Maps — Figures
display_outputs(out_root / "maps")

## 7. Metrics (Deterministic)

Consolidates scalar metrics (MAE, RMSE, Bias, Pearson R, FSS) into a single comparison table.
This allows for a direct ranking of model performance.

**Relevant Code:**
*   [`intercompare_deterministic_metrics`](../src/swissclim_evaluations/intercompare.py#L1185): Merges CSV files from different models into a combined DataFrame.

In [ ]:
# Metrics — Tables
display_outputs(out_root / "deterministic")

## 8. ETS
Quick comparison plots saved during intercompare (if any).

In [ ]:
# ETS — Figures & Tables
display_outputs(out_root / "ets")

## 9. Probabilistic

Compares probabilistic metrics (CRPS, PIT, Spread-Skill Ratio).
Crucial for evaluating ensemble prediction systems.

**Relevant Code:**
*   [`intercompare_probabilistic`](../src/swissclim_evaluations/intercompare.py#L1557):
    *   Overlays PIT histograms.
    *   Compares CRPS maps.
    *   Aggregates CRPS summary statistics.

In [ ]:
# Probabilistic — Figures & Tables
display_outputs(out_root / "probabilistic")

## 10. Vertical Profiles

Compares the vertical structure of errors (e.g., NMAE) across pressure levels.
Helps identify if models perform better or worse at specific altitudes (e.g., surface vs. upper atmosphere).

**Relevant Code:**
*   [`intercompare_vertical_profiles`](../src/swissclim_evaluations/intercompare.py#L196): Overlays error profiles from multiple models on the same plot.

In [ ]:
# Vertical Profiles — Figures & Tables
display_outputs(out_root / "vertical_profiles")

## 11. SSIM

This section compares the Structural Similarity Index Measure (SSIM) across models. SSIM is a perceptual metric that quantifies image quality degradation, measuring the structural similarity between the predicted and observed fields.

- **Range:** -1 to 1 (1 = perfect similarity)
- **Interpretation:** Higher values indicate better preservation of spatial structures and textures.
- **Average:** The plot includes an "average" category which represents the mean SSIM across all variables for each model.

In [ ]:
from IPython.display import Image, display
import pandas as pd

# Display SSIM comparison table
ssim_csv = out_root / "ssim" / "ssim_combined.csv"
if ssim_csv.exists():
    print(f"Loading SSIM metrics from: {ssim_csv}")
    df_ssim = pd.read_csv(ssim_csv)
    
    # Pivot to have variables as columns
    if "variable" in df_ssim.columns and "SSIM" in df_ssim.columns:
        df_pivot = df_ssim.pivot(index="model", columns="variable", values="SSIM")
        
        # Move AVERAGE_SSIM to the end if present
        if "AVERAGE_SSIM" in df_pivot.columns:
            cols = [c for c in df_pivot.columns if c != "AVERAGE_SSIM"] + ["AVERAGE_SSIM"]
            df_pivot = df_pivot[cols]
            
            # Display with style (bold the average column)
            styler = df_pivot.style.set_properties(subset=["AVERAGE_SSIM"], **{'font-weight': 'bold', 'border-left': '2px solid black'})
            display(styler)
        else:
            display(df_pivot)
    else:
        display(df_ssim)
else:
    print(f"No SSIM metrics combined CSV found at: {ssim_csv}")

# Display SSIM comparison plot
ssim_plot = out_root / "ssim" / "ssim_comparison.png"
if ssim_plot.exists():
    display(Image(filename=str(ssim_plot)))
else:
    print(f"No SSIM comparison plot found at: {ssim_plot}")